# 1. Project Overview

This notebook performs all dataset inspection, preprocessing, and feature engineering for a **binary structural damage classification** project using the [PEER Φ-Net Task 2](https://peer.berkeley.edu/) dataset.

## Goal
Prepare two parallel input pipelines for downstream model training:
1. **Classical ML branch** — HOG feature vectors for an SVM classifier.
2. **Deep learning branch** — Resized and scaled images for InceptionV3 fine-tuning.

## Dataset
- **Source**: PEER Φ-Net Task 2 (damage state classification)
- **Task 2_1**: `data/task2_damage_state_1/` — contains `X_train`, `y_train`, `X_test`, `y_test`
- **Task 2_2**: `data/task2_damage_state_2/` — contains `X_train` only

## Label Convention
Labels follow the PEER README specification:
- `0` = **Damaged** (D)
- `1` = **Undamaged** (UD)

## What This Notebook Produces
- A reproducible train/validation/test split (stratified, seed=42)
- Preprocessed images: `float32`, normalised to `[0.0, 1.0]`, 3 channels
- HOG feature arrays saved to `results/` (`X_train_hog.npy`, `X_val_hog.npy`, `X_test_hog.npy`)
- A fitted `StandardScaler` saved to `results/hog_scaler.pkl`
- InceptionV3-ready images: resized to `299×299`, scaled to `[-1.0, 1.0]`

> **Note**: No model training occurs in this notebook. The original `data/` files are never modified.

## 2. Imports

In [ ]:
import os
import pathlib

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
%matplotlib inline

import cv2

from skimage.feature import hog
from skimage import exposure

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.applications.inception_v3 import preprocess_input as inception_preprocess

import joblib

## 3. Dataset Paths & Configuration

In [ ]:
# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42

# ── Split parameters ──────────────────────────────────────────────────────────
VAL_FRACTION = 0.15          # fraction of training pool reserved for validation

# ── HOG feature extraction parameters ────────────────────────────────────────
HOG_PIXELS_PER_CELL = (8, 8)
HOG_CELLS_PER_BLOCK = (2, 2)
HOG_ORIENTATIONS    = 9

# ── InceptionV3 input parameters ─────────────────────────────────────────────
INCEPTION_SIZE = (299, 299)

# ── Batch size for memory-safe processing ─────────────────────────────────────
BATCH_SIZE = 256

# ── Directory paths ───────────────────────────────────────────────────────────
DATA_DIR    = pathlib.Path("../data")
RESULTS_DIR = pathlib.Path("../results")

# Create results directory if it does not exist
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"DATA_DIR    : {DATA_DIR.resolve()}")
print(f"RESULTS_DIR : {RESULTS_DIR.resolve()}")
print(f"SEED={SEED} | VAL_FRACTION={VAL_FRACTION} | BATCH_SIZE={BATCH_SIZE}")
print(f"HOG  pixels_per_cell={HOG_PIXELS_PER_CELL}  cells_per_block={HOG_CELLS_PER_BLOCK}  orientations={HOG_ORIENTATIONS}")
print(f"InceptionV3 input size: {INCEPTION_SIZE}")

## 4. Dataset Inspection

In [ ]:
# ── Task 2: File existence and size check ─────────────────────────────────────
#
# Define the 5 required dataset files. Paths are relative to the notebook
# location (notebooks/), so the data root is one level up (../data/).

REQUIRED_FILES = {
    "Task2_1 X_train": DATA_DIR / "task2_damage_state_1" / "task2_X_train.npy",
    "Task2_1 y_train": DATA_DIR / "task2_damage_state_1" / "task2_y_train.npy",
    "Task2_1 X_test" : DATA_DIR / "task2_damage_state_1" / "task2_X_test.npy",
    "Task2_1 y_test" : DATA_DIR / "task2_damage_state_1" / "task2_y_test.npy",
    "Task2_2 X_train": DATA_DIR / "task2_damage_state_2" / "task2_X_train.npy",
}

# --- Existence check ---
missing = [str(path) for path in REQUIRED_FILES.values() if not path.exists()]

if missing:
    raise RuntimeError(
        "The following required dataset files are missing:\n"
        + "\n".join(f"  - {p}" for p in missing)
    )

print("All required files are present.\n")

# --- File size display (MB, 2 decimal places) ---
print(f"{'Label':<20}  {'Path':<65}  {'Size (MB)':>10}")
print("-" * 100)
for label, path in REQUIRED_FILES.items():
    size_mb = path.stat().st_size / (1024 ** 2)
    print(f"{label:<20}  {str(path):<65}  {size_mb:>10.2f}")

In [ ]:
# ── Task 3: Safe array shape and dtype inspection ─────────────────────────────
#
# task2_X_train.npy (Task 2_1) is potentially very large (~7 GB).
# Load it with mmap_mode='r' so only metadata and indexed rows are read
# from disk on demand — the full buffer never enters RAM.
# All other arrays are small and loaded normally.

def inspect_array(name, arr):
    """Print shape, dtype, min, max and derive spatial/channel dimensions."""
    arr_min = float(arr.min())
    arr_max = float(arr.max())

    print(f"\n{'─' * 60}")
    print(f"  Array : {name}")
    print(f"  Shape : {arr.shape}")
    print(f"  dtype : {arr.dtype}")
    print(f"  min   : {arr_min}")
    print(f"  max   : {arr_max}")

    # Derive N, H, W, C from shape
    if arr.ndim == 4:
        N, H, W, C = arr.shape
        print(f"  dims  : N={N}  H={H}  W={W}  C={C}")
    elif arr.ndim == 3:
        N, H, W = arr.shape
        C = 1  # single channel, no channel axis present
        print(f"  dims  : N={N}  H={H}  W={W}  C={C} (3-D array, channel axis absent)")
    else:
        # 1-D or 2-D — label array or flat array
        print(f"  dims  : {arr.ndim}-D array (label / flat)")

    # Pixel range validation for image arrays only (ndim >= 3)
    if arr.ndim >= 3:
        is_uint8_range  = (arr_min >= 0)   and (arr_max <= 255)
        is_float_range  = (arr_min >= 0.0) and (arr_max <= 1.0)
        # Warn only when the range is neither [0,255] nor [0.0,1.0]
        if not (is_uint8_range or is_float_range):
            print(f"  ⚠ WARNING: Unexpected pixel range for '{name}': "
                  f"min={arr_min}, max={arr_max} "
                  f"(expected [0, 255] or [0.0, 1.0])")
        else:
            detected = "[0, 255] (uint8-scale)" if arr_max > 1.0 else "[0.0, 1.0] (float-scale)"
            print(f"  range : {detected} ✓")


# Load Task 2_1 X_train with memory mapping (large file — do NOT load into RAM)
print("Loading Task 2_1 X_train with mmap_mode='r' (memory-mapped, large file) ...")
X_train_21 = np.load(
    DATA_DIR / "task2_damage_state_1" / "task2_X_train.npy",
    mmap_mode="r"
)

# Load remaining arrays normally (they are small)
print("Loading remaining arrays normally ...")
y_train_21 = np.load(DATA_DIR / "task2_damage_state_1" / "task2_y_train.npy")
X_test_21  = np.load(DATA_DIR / "task2_damage_state_1" / "task2_X_test.npy")
y_test_21  = np.load(DATA_DIR / "task2_damage_state_1" / "task2_y_test.npy")
X_train_22 = np.load(DATA_DIR / "task2_damage_state_2" / "task2_X_train.npy")

print("\n" + "═" * 60)
print("  Array Shape / dtype / Range Inspection")
print("═" * 60)

inspect_array("Task2_1  X_train (mmap)", X_train_21)
inspect_array("Task2_1  y_train",        y_train_21)
inspect_array("Task2_1  X_test",         X_test_21)
inspect_array("Task2_1  y_test",         y_test_21)
inspect_array("Task2_2  X_train",        X_train_22)

print(f"\n{'─' * 60}")

# --- Sample count consistency checks (X vs y) --------------------------------
# For Task 2_1 train set
print("\n── Sample count consistency checks ──")
if X_train_21.shape[0] != y_train_21.shape[0]:
    print(
        f"  ✗ MISMATCH (Task 2_1 train): "
        f"X_train has {X_train_21.shape[0]} samples but "
        f"y_train has {y_train_21.shape[0]} samples."
    )
else:
    print(f"  ✓ Task 2_1 train : X and y both have {X_train_21.shape[0]} samples.")

# For Task 2_1 test set
if X_test_21.shape[0] != y_test_21.shape[0]:
    print(
        f"  ✗ MISMATCH (Task 2_1 test): "
        f"X_test has {X_test_21.shape[0]} samples but "
        f"y_test has {y_test_21.shape[0]} samples."
    )
else:
    print(f"  ✓ Task 2_1 test  : X and y both have {X_test_21.shape[0]} samples.")

In [ ]:
# ── Task 4: Class distribution analysis ──────────────────────────────────────
#
# Compute class counts, percentages, and imbalance ratio for BOTH
# the training and test label arrays.
# Labels: 0 = Damaged (D), 1 = Undamaged (UD)  [PEER README convention]

LABEL_NAMES = {0: "Damaged", 1: "Undamaged"}


def class_distribution_report(split_name, y_labels, x_samples):
    """
    Print class counts, percentages, and imbalance ratio for a label array.

    Parameters
    ----------
    split_name : str   — display name (e.g. 'Train', 'Test')
    y_labels   : ndarray — 1-D integer label array
    x_samples  : ndarray — corresponding image array (used for count check only)
    """
    print(f"\n{'═' * 55}")
    print(f"  Class Distribution — {split_name}")
    print(f"{'═' * 55}")

    # --- Sample count consistency check --------------------------------------
    if y_labels.shape[0] != x_samples.shape[0]:
        print(
            f"  ✗ ERROR: Sample count mismatch — "
            f"y has {y_labels.shape[0]} rows but X has {x_samples.shape[0]} rows. "
            f"Skipping distribution analysis for {split_name}."
        )
        return

    total = len(y_labels)
    unique_classes = np.unique(y_labels)

    counts = {}
    for cls in unique_classes:
        counts[int(cls)] = int(np.sum(y_labels == cls))

    # Print header row
    print(f"  {'Label':<6}  {'Class':<12}  {'Count':>8}  {'Percentage':>12}")
    print(f"  {'─'*6}  {'─'*12}  {'─'*8}  {'─'*12}")

    for cls_id, count in sorted(counts.items()):
        class_name = LABEL_NAMES.get(cls_id, f"Unknown({cls_id})")
        pct = 100.0 * count / total
        print(f"  {cls_id:<6}  {class_name:<12}  {count:>8}  {pct:>11.2f}%")

    print(f"  {'─'*6}  {'─'*12}  {'─'*8}  {'─'*12}")
    print(f"  {'Total':<6}  {'':12}  {total:>8}  {'100.00%':>12}")

    # --- Imbalance ratio -----------------------------------------------------
    if len(unique_classes) < 2:
        # Only one class present — skip ratio to avoid division by zero
        cls_id = int(unique_classes[0])
        cls_name = LABEL_NAMES.get(cls_id, f"Unknown({cls_id})")
        print(
            f"\n  ⚠ WARNING: Only one class present (label={cls_id}, '{cls_name}'). "
            f"Imbalance ratio cannot be computed for {split_name}."
        )
    else:
        majority_count = max(counts.values())
        minority_count = min(counts.values())
        ratio = majority_count / minority_count
        majority_label = max(counts, key=counts.get)
        minority_label = min(counts, key=counts.get)
        majority_name  = LABEL_NAMES.get(majority_label, str(majority_label))
        minority_name  = LABEL_NAMES.get(minority_label, str(minority_label))
        print(
            f"\n  Imbalance ratio : {ratio:.2f}  "
            f"(majority='{majority_name}' {majority_count}  /  "
            f"minority='{minority_name}' {minority_count})"
        )


# Run reports for both splits
class_distribution_report("Train (Task 2_1)", y_train_21, X_train_21)
class_distribution_report("Test  (Task 2_1)", y_test_21,  X_test_21)

## 5. Task 2_1 vs Task 2_2 Comparison

In [ ]:
# ── Task 5: Task 2_1 vs Task 2_2 Comparison ──────────────────────────────────
#
# Compare the two dataset variants on key attributes so we can make an
# informed decision about which to use for downstream modelling.
#
# Variables already in scope from Section 4:
#   X_train_21, y_train_21, X_test_21, y_test_21  (Task 2_1)
#   X_train_22                                     (Task 2_2 — X only, no labels)

import os

# ── Helper: file size in MB ───────────────────────────────────────────────────
def _file_mb(path):
    """Return file size in MB, or None if the file does not exist."""
    try:
        return path.stat().st_size / (1024 ** 2)
    except FileNotFoundError:
        return None

# ── Helper: spatial/channel dims from an ndarray ─────────────────────────────
def _dims(arr):
    """Return (H, W, C) from a (N,H,W,C) or (N,H,W) array, or None."""
    if arr is None:
        return None, None, None
    if arr.ndim == 4:
        _, H, W, C = arr.shape
        return H, W, C
    if arr.ndim == 3:
        _, H, W = arr.shape
        return H, W, 1
    return None, None, None

# ── Helper: available splits ──────────────────────────────────────────────────
# A split is "available" only when BOTH the X and y .npy files are present.
task21_dir = DATA_DIR / "task2_damage_state_1"
task22_dir = DATA_DIR / "task2_damage_state_2"

def _available_splits(task_dir):
    splits = []
    for split in ["train", "test"]:
        x_path = task_dir / f"task2_X_{split}.npy"
        y_path = task_dir / f"task2_y_{split}.npy"
        if x_path.exists() and y_path.exists():
            splits.append(split)
    return splits if splits else ["none"]

splits_21 = _available_splits(task21_dir)
splits_22 = _available_splits(task22_dir)

# ── Gather per-dataset attributes ────────────────────────────────────────────
# Task 2_1 — X_train_21, y_train_21, X_test_21, y_test_21 already loaded
x21_path   = task21_dir / "task2_X_train.npy"
x22_path   = task22_dir / "task2_X_train.npy"

mb_21      = _file_mb(x21_path)
mb_22      = _file_mb(x22_path)

shape_21   = X_train_21.shape
shape_22   = X_train_22.shape

H21, W21, C21 = _dims(X_train_21)
H22, W22, C22 = _dims(X_train_22)

dtype_21   = str(X_train_21.dtype)
dtype_22   = str(X_train_22.dtype)

pmin_21    = float(X_train_21.min())
pmax_21    = float(X_train_21.max())
pmin_22    = float(X_train_22.min())
pmax_22    = float(X_train_22.max())

# Label sets — Task 2_1 has labels; Task 2_2 has no y files
labels_21  = sorted(map(int, set(y_train_21.tolist()) | set(y_test_21.tolist())))
labels_22  = "N/A (no y files)"

mb_21_str  = f"{mb_21:.2f} MB" if mb_21 is not None else "file not found"
mb_22_str  = f"{mb_22:.2f} MB" if mb_22 is not None else "file not found"

# ── Print comparison table ────────────────────────────────────────────────────
col_w   = 30
val_w   = 28
sep     = "─" * (col_w + val_w * 2 + 7)

print(sep)
print(f"{'Attribute':<{col_w}}  {'Task_2_1':<{val_w}}  {'Task_2_2':<{val_w}}")
print(sep)

rows = [
    ("X_train file size",        mb_21_str,                       mb_22_str),
    ("X_train shape",            str(shape_21),                   str(shape_22)),
    ("Spatial dims (H × W)",     f"{H21} × {W21}",                f"{H22} × {W22}"),
    ("Channels (C)",             str(C21),                        str(C22)),
    ("dtype",                    dtype_21,                        dtype_22),
    ("Pixel min",                f"{pmin_21:.4f}",                 f"{pmin_22:.4f}"),
    ("Pixel max",                f"{pmax_21:.4f}",                 f"{pmax_22:.4f}"),
    ("Available splits",         ", ".join(splits_21),             ", ".join(splits_22)),
    ("Label set",                str(labels_21),                  str(labels_22)),
]

for attr, v21, v22 in rows:
    print(f"  {attr:<{col_w-2}}  {v21:<{val_w}}  {v22:<{val_w}}")

print(sep)

# ── Compatibility warnings ────────────────────────────────────────────────────
warnings_issued = False

if (H21, W21) != (H22, W22):
    print(f"\n⚠  WARNING: Spatial dimensions differ "
          f"(Task_2_1: {H21}×{W21}  vs  Task_2_2: {H22}×{W22}). "
          "Combining datasets is NOT recommended.")
    warnings_issued = True

if C21 != C22:
    print(f"\n⚠  WARNING: Channel counts differ "
          f"(Task_2_1: {C21}  vs  Task_2_2: {C22}). "
          "Combining datasets is NOT recommended.")
    warnings_issued = True

# Task 2_2 has no labels — inherently incompatible for supervised training
if "none" in splits_22 or not any(s in splits_22 for s in ["train", "test"]):
    print("\n⚠  WARNING: Task_2_2 has no y (label) files. "
          "Label sets differ between datasets — combining is NOT recommended.")
    warnings_issued = True

if not warnings_issued:
    print("\n✓  No compatibility warnings between Task_2_1 and Task_2_2.")

# ── Selected Dataset ──────────────────────────────────────────────────────────
print("\n")
print("#### Selected Dataset")
print("-" * 50)
print(
    "Selected: Task_2_1\n"
    "\n"
    "Justification: Task_2_1 provides official label arrays for both the train "
    "and test splits (available splits: train, test), enabling fully supervised "
    "binary classification with stratified cross-validation and unbiased test-set "
    "evaluation. Task_2_2 supplies only X_train with no corresponding y files "
    "(available splits: none), making it unusable for supervised training or "
    "evaluation without additional annotation effort. Therefore Task_2_1 is "
    "selected as the dataset for all downstream preprocessing and modelling steps."
)

SELECTED_DATASET = "Task_2_1"
print(f"\nSELECTED_DATASET = {SELECTED_DATASET!r}")

## 6. Class Distribution

In [ ]:
# ── Task 6: Class Distribution Bar Chart ──────────────────────────────────────
#
# This cell shows the PRE-SPLIT class distribution for the training pool
# (y_train_21) and the official test set (y_test_21).
#
# Note: The post-split bar chart (train/val/test after Section 8) can be
# produced by re-running this cell after the split cell has been executed —
# simply swap the y arrays below for y_train_split / y_val_split / y_test_split.

# ── Compute counts ────────────────────────────────────────────────────────────
class_ids   = sorted(LABEL_NAMES.keys())           # [0, 1]
class_names = [LABEL_NAMES[c] for c in class_ids]  # ['Damaged', 'Undamaged']

splits_data = {
    "Train pool (Task 2_1)": y_train_21,
    "Test  (Task 2_1)"     : y_test_21,
}

# Build count matrix: rows = splits, cols = classes
split_names  = list(splits_data.keys())
count_matrix = [
    [int(np.sum(y == c)) for c in class_ids]
    for y in splits_data.values()
]

# ── Print table ───────────────────────────────────────────────────────────────
hdr_w = max(len(n) for n in split_names) + 2
print(f"{'Split':<{hdr_w}}  {'Damaged (0)':>14}  {'Undamaged (1)':>14}  {'Total':>10}")
print("-" * (hdr_w + 46))
for name, counts in zip(split_names, count_matrix):
    total = sum(counts)
    pcts  = [f"{c/total*100:.1f}%" for c in counts]
    print(f"{name:<{hdr_w}}  {counts[0]:>8} ({pcts[0]:>6})  {counts[1]:>8} ({pcts[1]:>6})  {total:>10}")

# ── Bar chart ─────────────────────────────────────────────────────────────────
import numpy as _np_local

x      = np.arange(len(class_names))   # [0, 1]
n_sp   = len(split_names)
width  = 0.35
offsets = np.linspace(-(n_sp - 1) * width / 2,
                       (n_sp - 1) * width / 2,
                       n_sp)

colors = ["#4C72B0", "#DD8452"]  # blue for train, orange for test

fig, ax = plt.subplots(figsize=(10, 5))

for i, (name, counts, offset) in enumerate(
        zip(split_names, count_matrix, offsets)):
    bars = ax.bar(x + offset, counts, width=width,
                  label=name, color=colors[i % len(colors)], alpha=0.85)
    ax.bar_label(bars, fmt="%d", padding=3, fontsize=9)

ax.set_title("Class Distribution", fontsize=14, pad=12)
ax.set_xlabel("Class", fontsize=11)
ax.set_ylabel("Sample Count", fontsize=11)
ax.set_xticks(x)
ax.set_xticklabels(class_names, fontsize=11)
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{int(v):,}"))
ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()
print("\nNote: Re-run this cell after Section 8 (split) to see per-split "
      "(train/val/test) distribution using y_train_split, y_val_split, y_test_split.")

## 7. Sample Image Visualization

In [ ]:
# ── Task 7: Sample Image Visualization ───────────────────────────────────────
#
# Display 3 representative images per class from the full training pool
# (y_train_21) using fixed-seed sampling. We use ALL training indices here
# (not train_idx from the split) because the split is defined in Section 8
# which follows this cell in notebook order.
#
# Images are accessed via indexed read on the mmap array — no full copy.

N_SAMPLES_PER_CLASS = 3
rng = np.random.default_rng(SEED)

# ── Select indices per class ──────────────────────────────────────────────────
all_train_indices = np.arange(len(y_train_21))
selected = {}   # class_id → list of array indices

for cls_id, cls_name in LABEL_NAMES.items():
    cls_indices = all_train_indices[y_train_21 == cls_id]

    if len(cls_indices) == 0:
        print(f"⚠  WARNING: No samples found for class '{cls_name}' (label={cls_id}). "
              "Skipping this class in the visualization grid.")
        selected[cls_id] = []
        continue

    n_available = len(cls_indices)
    n_pick      = min(N_SAMPLES_PER_CLASS, n_available)

    if n_available < N_SAMPLES_PER_CLASS:
        print(f"⚠  WARNING: Class '{cls_name}' (label={cls_id}) has only "
              f"{n_available} sample(s); showing {n_pick} instead of "
              f"{N_SAMPLES_PER_CLASS}.")

    chosen = rng.choice(cls_indices, size=n_pick, replace=False)
    selected[cls_id] = sorted(chosen.tolist())

# ── Build display order: rows = classes, cols = samples ──────────────────────
# Determine actual grid dimensions from available samples.
n_rows = len([v for v in selected.values() if len(v) > 0])
n_cols = max((len(v) for v in selected.values()), default=0)

if n_rows == 0 or n_cols == 0:
    print("No samples available to display.")
else:
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(max(10, n_cols * 3.5),
                                      max(6,  n_rows * 3.5)))

    # Normalise axes to always be a 2-D list for uniform indexing
    if n_rows == 1 and n_cols == 1:
        axes = [[axes]]
    elif n_rows == 1:
        axes = [list(axes)]
    elif n_cols == 1:
        axes = [[ax] for ax in axes]
    else:
        axes = [list(row) for row in axes]

    row_ptr = 0
    for cls_id in sorted(LABEL_NAMES.keys()):
        cls_name    = LABEL_NAMES[cls_id]
        idx_list    = selected.get(cls_id, [])

        if not idx_list:
            continue

        # Read only the needed rows from the mmap array
        images = X_train_21[idx_list]   # shape: (k, H, W, C) or (k, H, W)

        for col_ptr, (arr_idx, img) in enumerate(zip(idx_list, images)):
            ax = axes[row_ptr][col_ptr]

            # Handle display: squeeze channel dim if grayscale (C=1)
            if img.ndim == 3 and img.shape[-1] == 1:
                display_img = img.squeeze(-1)
                ax.imshow(display_img, cmap="gray")
            elif img.ndim == 2:
                ax.imshow(img, cmap="gray")
            else:
                # Clip to uint8 range for display if needed
                if img.dtype != np.uint8:
                    display_img = np.clip(img, 0, 1)
                else:
                    display_img = img
                ax.imshow(display_img)

            ax.set_title(f"{cls_name}\nidx={arr_idx}", fontsize=9)
            ax.axis("off")

        # Hide any unused column slots in this row
        for col_ptr in range(len(idx_list), n_cols):
            axes[row_ptr][col_ptr].axis("off")

        row_ptr += 1

    plt.suptitle("Representative Sample Images", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

## 8. Train/Validation/Test Split

In [ ]:
# ── Task 8: Train / Validation / Test Split ───────────────────────────────────
#
# Produces stratified train/val index arrays from the official Task 2_1
# training pool. The official test set is used unchanged as the test split.
#
# Outputs
# -------
# train_idx     : ndarray of int  — indices into X_train_21 / y_train_21
# val_idx       : ndarray of int  — indices into X_train_21 / y_train_21
# y_train_split : ndarray — labels for training subset
# y_val_split   : ndarray — labels for validation subset
# y_test_split  : ndarray — labels for test set (= y_test_21)

# ── Validate VAL_FRACTION ─────────────────────────────────────────────────────
if not (0.05 < VAL_FRACTION < 0.50):
    raise ValueError(
        f"VAL_FRACTION={VAL_FRACTION!r} is outside the valid range (0.05, 0.50) "
        f"exclusive. Please update the configuration cell and re-run."
    )

# ── Create stratified split ───────────────────────────────────────────────────
pool_indices = np.arange(len(y_train_21))

train_idx, val_idx = train_test_split(
    pool_indices,
    test_size=VAL_FRACTION,
    stratify=y_train_21,
    random_state=SEED,
)

# ── Store label sub-arrays ────────────────────────────────────────────────────
y_train_split = y_train_21[train_idx]
y_val_split   = y_train_21[val_idx]
y_test_split  = y_test_21

# ── Per-split reporting helper ────────────────────────────────────────────────
def _split_report(split_name, y_labels, n_total_check=None):
    """Print sample count, per-class counts, and per-class percentages."""
    n      = len(y_labels)
    cls_ids = sorted(LABEL_NAMES.keys())

    print(f"\n  ── {split_name} ({'n=' + str(n)}) ──")
    print(f"    {'Class':<14}  {'Count':>8}  {'Percentage':>12}")
    print(f"    {'─'*14}  {'─'*8}  {'─'*12}")

    for cls_id in cls_ids:
        count     = int(np.sum(y_labels == cls_id))
        pct       = 100.0 * count / n if n > 0 else 0.0
        cls_name  = LABEL_NAMES.get(cls_id, str(cls_id))
        print(f"    {cls_name:<14}  {count:>8}  {pct:>11.2f}%")

    print(f"    {'─'*14}  {'─'*8}  {'─'*12}")
    print(f"    {'Total':<14}  {n:>8}  {'100.00%':>12}")

# ── Print summary ─────────────────────────────────────────────────────────────
print("═" * 55)
print("  Train / Validation / Test Split Summary")
print("═" * 55)
print(f"  Training pool size : {len(pool_indices)}")
print(f"  VAL_FRACTION       : {VAL_FRACTION}")
print(f"  SEED               : {SEED}")

_split_report("Train",      y_train_split)
_split_report("Validation", y_val_split)
_split_report("Test",       y_test_split)

print(f"\n  train_idx : {len(train_idx)} indices")
print(f"  val_idx   : {len(val_idx)} indices")
print(f"  test set  : {len(y_test_split)} samples (official Task 2_1 test split)")
print("═" * 55)

## 9. Image Preprocessing

In [ ]:
# ── Task 9: Image Preprocessing Pipeline ─────────────────────────────────────
#
# Converts raw images to float32, normalises to [0.0, 1.0], and enforces
# exactly 3 channels. Processes in batches of BATCH_SIZE to keep RAM usage
# bounded — the original ~7 GB mmap array is never fully copied into RAM.
#
# No data augmentation is applied.
# Original data/  files are never modified.
#
# Outputs
# -------
# X_train_proc : np.ndarray float32  (len(train_idx), H, W, 3)
# X_val_proc   : np.ndarray float32  (len(val_idx),   H, W, 3)
# X_test_proc  : np.ndarray float32  (len(X_test_21), H, W, 3)

import warnings as _warnings


def preprocess_batch(batch):
    """
    Preprocess a batch of images.

    Parameters
    ----------
    batch : np.ndarray
        A slice of the source array, shape (k, H, W, C) or (k, H, W).
        May be a mmap sub-array; a copy is made internally.

    Returns
    -------
    processed : list of np.ndarray
        Each element is a (H, W, 3) float32 array, or None if the sample
        was skipped (unsupported dtype or channel count > 3).
    skip_flags : list of bool
        True where the corresponding sample was skipped.
    """
    # Make an in-RAM copy so we never mutate the mmap buffer
    batch = np.array(batch)
    source_dtype = batch.dtype

    processed  = []
    skip_flags = []

    for local_idx in range(len(batch)):
        img = batch[local_idx]   # shape: (H, W, C) or (H, W)

        # ── Add channel dim if missing ────────────────────────────────────
        if img.ndim == 2:
            img = img[:, :, np.newaxis]   # (H, W) → (H, W, 1)

        C = img.shape[-1]

        # ── Channel validation ────────────────────────────────────────────
        if C > 3:
            print(f"    [SKIP] batch-local index {local_idx}: "
                  f"{C} channels (>3) — skipping sample.")
            processed.append(None)
            skip_flags.append(True)
            continue

        # ── dtype-aware normalisation ─────────────────────────────────────
        if source_dtype == np.uint8:
            img = img.astype(np.float32) / 255.0
        elif source_dtype == np.float64:
            img = img.astype(np.float32)   # already [0, 1], just cast
        elif source_dtype == np.float32:
            img = img.copy()               # already float32 — no rescaling
        else:
            print(f"    [WARN] batch-local index {local_idx}: "
                  f"unsupported dtype '{source_dtype}' — skipping sample.")
            processed.append(None)
            skip_flags.append(True)
            continue

        # ── Channel replication: 1 → 3 ────────────────────────────────────
        if C == 1:
            img = np.repeat(img, 3, axis=-1)   # (H, W, 1) → (H, W, 3)
        # C == 3: keep as-is

        # ── Post-normalisation range check ────────────────────────────────
        pmin, pmax = float(img.min()), float(img.max())
        if pmin < 0.0 or pmax > 1.0:
            print(f"    [WARN] batch-local index {local_idx}: "
                  f"pixel values out of [0,1] after normalisation "
                  f"(min={pmin:.4f}, max={pmax:.4f}).")

        processed.append(img)
        skip_flags.append(False)

    return processed, skip_flags


def process_split(source_array, indices, split_name):
    """
    Apply preprocess_batch to all samples identified by *indices* in
    *source_array*, collecting results into a single float32 ndarray.

    Parameters
    ----------
    source_array : np.ndarray (may be mmap)
        Full image array (N, H, W, C).
    indices : array-like of int
        Positional indices to select from source_array.
    split_name : str
        Display label for progress messages.

    Returns
    -------
    result : np.ndarray, float32, shape (len(valid_indices), H, W, 3)
    """
    indices      = np.asarray(indices)
    n_total      = len(indices)
    result_list  = []
    n_skipped    = 0

    print(f"  Processing '{split_name}': {n_total} samples "
          f"in batches of {BATCH_SIZE} ...")

    for batch_start in range(0, n_total, BATCH_SIZE):
        batch_end     = min(batch_start + BATCH_SIZE, n_total)
        batch_indices = indices[batch_start:batch_end]

        # Indexed read from mmap — only these rows enter RAM
        batch_raw = source_array[batch_indices]

        processed, skip_flags = preprocess_batch(batch_raw)

        for img, skipped in zip(processed, skip_flags):
            if not skipped:
                result_list.append(img)
            else:
                n_skipped += 1

        samples_done = batch_end
        if samples_done % 1000 < BATCH_SIZE or samples_done == n_total:
            print(f"    ... {samples_done}/{n_total} samples processed.")

    if n_skipped > 0:
        print(f"  ⚠  '{split_name}': {n_skipped} sample(s) skipped due to "
              "unsupported dtype or channel count.")

    if len(result_list) == 0:
        raise RuntimeError(
            f"All samples in split '{split_name}' were skipped. "
            "Check dtype and channel count of the source array."
        )

    result = np.stack(result_list, axis=0)   # (n_valid, H, W, 3)
    print(f"  '{split_name}' done → shape={result.shape}, dtype={result.dtype}")
    return result


# ── Run preprocessing for all three splits ────────────────────────────────────
print("═" * 60)
print("  Image Preprocessing Pipeline")
print("═" * 60)

X_train_proc = process_split(X_train_21, train_idx,                  "Train")
X_val_proc   = process_split(X_train_21, val_idx,                    "Validation")
X_test_proc  = process_split(X_test_21,  np.arange(len(X_test_21)), "Test")

# ── Final shapes and dtype summary ───────────────────────────────────────────
print("\n" + "─" * 60)
print("  Preprocessing Complete — Final Array Summary")
print("─" * 60)
print(f"  X_train_proc : shape={X_train_proc.shape}  dtype={X_train_proc.dtype}")
print(f"  X_val_proc   : shape={X_val_proc.shape}  dtype={X_val_proc.dtype}")
print(f"  X_test_proc  : shape={X_test_proc.shape}  dtype={X_test_proc.dtype}")
print("─" * 60)
print(f"  Pixel range (train) : min={X_train_proc.min():.4f}  "
      f"max={X_train_proc.max():.4f}")

## 10. Feature Engineering for SVM

### Why HOG for Structural Damage Detection

HOG (Histogram of Oriented Gradients) captures local gradient orientations that encode edge and texture patterns — precisely the kind of information that distinguishes cracked or damaged structural surfaces from undamaged ones. Damage signatures such as concrete cracks, spalling, and rebar exposure manifest as directional intensity changes that HOG encodes compactly regardless of small lighting variations.

By dividing the image into local cells and normalising gradients block-by-block, HOG achieves robustness to uniform illumination shifts and small geometric deformations, both of which are common in field-captured structural images. This makes HOG a practical and interpretable feature choice for the classical SVM branch.

In [ ]:
# ── Task 10: HOG Feature Extraction and Scaling ───────────────────────────────
#
# Extracts HOG feature vectors from preprocessed images (batch-by-batch).
# Fits StandardScaler on training features only; transforms val and test.
# Saves arrays and scaler to results/.

from skimage.feature import hog as skimage_hog

# ── Print HOG configuration ───────────────────────────────────────────────────
print("═" * 60)
print("  HOG Feature Extraction Configuration")
print("═" * 60)
print(f"  pixels_per_cell : {HOG_PIXELS_PER_CELL}")
print(f"  cells_per_block : {HOG_CELLS_PER_BLOCK}")
print(f"  orientations    : {HOG_ORIENTATIONS}")


def extract_hog_features(X_proc, split_name):
    """
    Extract HOG feature vectors from a preprocessed image array batch-by-batch.

    Parameters
    ----------
    X_proc     : np.ndarray float32, shape (N, H, W, 3)
    split_name : str — display name for progress messages

    Returns
    -------
    features : np.ndarray float64, shape (N, D)
    """
    n_total      = len(X_proc)
    feature_list = []

    print(f"  Extracting HOG features for '{split_name}': {n_total} samples ...")

    for batch_start in range(0, n_total, BATCH_SIZE):
        batch_end = min(batch_start + BATCH_SIZE, n_total)
        batch     = X_proc[batch_start:batch_end]

        for img in batch:
            feat = skimage_hog(
                img,
                pixels_per_cell=HOG_PIXELS_PER_CELL,
                cells_per_block=HOG_CELLS_PER_BLOCK,
                orientations=HOG_ORIENTATIONS,
                channel_axis=-1,
                feature_vector=True,
            )
            feature_list.append(feat)

        if batch_end % 1000 < BATCH_SIZE or batch_end == n_total:
            print(f"    ... {batch_end}/{n_total} samples done.")

    features = np.stack(feature_list, axis=0)
    print(f"  '{split_name}' HOG done → shape={features.shape}")
    return features


# ── Extract HOG features for all splits ──────────────────────────────────────
X_train_hog = extract_hog_features(X_train_proc, "Train")
X_val_hog   = extract_hog_features(X_val_proc,   "Validation")
X_test_hog  = extract_hog_features(X_test_proc,  "Test")

# ── Feature vector dimensionality ────────────────────────────────────────────
hog_feature_dim = int(X_train_hog.shape[1])
print(f"\n  HOG feature vector dimensionality: {hog_feature_dim}")

# ── Fit StandardScaler on training set ONLY ───────────────────────────────────
print("\n  Fitting StandardScaler on training HOG features ...")
hog_scaler = StandardScaler()
X_train_hog_scaled = hog_scaler.fit_transform(X_train_hog)
X_val_hog_scaled   = hog_scaler.transform(X_val_hog)
X_test_hog_scaled  = hog_scaler.transform(X_test_hog)
print("  StandardScaler fitted on training set only. Val and test transformed.")

# ── HOG visualisation for first training sample ───────────────────────────────
print("\n  Generating HOG visualisation for first training sample ...")
_, hog_image = skimage_hog(
    X_train_proc[0],
    pixels_per_cell=HOG_PIXELS_PER_CELL,
    cells_per_block=HOG_CELLS_PER_BLOCK,
    orientations=HOG_ORIENTATIONS,
    channel_axis=-1,
    feature_vector=True,
    visualize=True,
)
hog_image_rescaled = exposure.rescale_intensity(hog_image, in_range=(0, 10))

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(np.clip(X_train_proc[0], 0, 1))
axes[0].set_title("Original Image (train[0])", fontsize=11)
axes[0].axis("off")
axes[1].imshow(hog_image_rescaled, cmap="gray")
axes[1].set_title("HOG Visualisation (train[0])", fontsize=11)
axes[1].axis("off")
plt.suptitle("HOG Feature Visualisation", fontsize=13)
plt.tight_layout()
plt.show()

# ── Save scaled arrays and scaler to results/ ─────────────────────────────────
print("\n  Saving HOG arrays and scaler to results/ ...")
np.save(RESULTS_DIR / "X_train_hog.npy", X_train_hog_scaled)
np.save(RESULTS_DIR / "X_val_hog.npy",   X_val_hog_scaled)
np.save(RESULTS_DIR / "X_test_hog.npy",  X_test_hog_scaled)
joblib.dump(hog_scaler, RESULTS_DIR / "hog_scaler.pkl")

print(f"  Saved: results/X_train_hog.npy  → shape={X_train_hog_scaled.shape}")
print(f"  Saved: results/X_val_hog.npy    → shape={X_val_hog_scaled.shape}")
print(f"  Saved: results/X_test_hog.npy   → shape={X_test_hog_scaled.shape}")
print(f"  Saved: results/hog_scaler.pkl")
print("═" * 60)

## 11. InceptionV3 Preprocessing

### InceptionV3 Preprocessing

InceptionV3 expects inputs of shape `(299, 299, 3)` with pixel values in `[-1.0, 1.0]`.

**Resize method**: Bilinear interpolation (`cv2.INTER_LINEAR`) is used to resize images to `299 × 299`. Bilinear interpolation is preferred here because it produces smooth, continuous-valued output that avoids the aliasing artefacts of nearest-neighbour interpolation when upsampling small images.

**Pixel scaling**: `tf.keras.applications.inception_v3.preprocess_input` linearly maps float32 values in `[0.0, 1.0]` to `[-1.0, 1.0]`, matching the scale used during InceptionV3's original training on ImageNet.

No model training, fine-tuning, or inference with InceptionV3 occurs in this cell.
No augmentation is applied.

In [ ]:
# ── Task 11: InceptionV3 Data Preparation ────────────────────────────────────
#
# Resizes preprocessed images to INCEPTION_SIZE (299, 299) using bilinear
# interpolation and scales pixel values to [-1.0, 1.0] using
# tf.keras.applications.inception_v3.preprocess_input.
#
# No model training, fine-tuning, or inference occurs here.
# No augmentation is applied.
#
# Outputs (NOT saved to disk — too large):
#   X_train_inc : float32, shape (n_train, 299, 299, 3)
#   X_val_inc   : float32, shape (n_val,   299, 299, 3)
#   X_test_inc  : float32, shape (n_test,  299, 299, 3)

def prepare_inception_split(X_proc, split_name):
    """
    Resize images to INCEPTION_SIZE and scale pixels to [-1.0, 1.0].

    Parameters
    ----------
    X_proc     : np.ndarray float32, shape (N, H, W, 3), values in [0, 1]
    split_name : str — display label for progress messages

    Returns
    -------
    result : np.ndarray float32, shape (N, 299, 299, 3), values in [-1, 1]
    """
    n_total     = len(X_proc)
    target_h, target_w = INCEPTION_SIZE
    result      = np.empty((n_total, target_h, target_w, 3), dtype=np.float32)

    print(f"  Preparing InceptionV3 input for '{split_name}': {n_total} samples ...")

    for batch_start in range(0, n_total, BATCH_SIZE):
        batch_end = min(batch_start + BATCH_SIZE, n_total)
        batch     = X_proc[batch_start:batch_end]   # (k, H, W, 3)

        for i, img in enumerate(batch):
            # Resize to (299, 299) using bilinear interpolation
            resized = cv2.resize(
                img,
                (target_w, target_h),
                interpolation=cv2.INTER_LINEAR,
            )   # shape: (299, 299, 3), float32 in [0, 1]
            result[batch_start + i] = resized

        if batch_end % 1000 < BATCH_SIZE or batch_end == n_total:
            print(f"    ... {batch_end}/{n_total} samples resized.")

    # Apply InceptionV3 preprocessing: [0, 1] → [-1, 1]
    # inception_preprocess operates in-place on float arrays;
    # pass a copy to avoid mutating result in unexpected ways.
    result = inception_preprocess(result.copy())

    print(f"  '{split_name}' InceptionV3 prep done → shape={result.shape}, "
          f"dtype={result.dtype}, "
          f"pixel range=[{result.min():.4f}, {result.max():.4f}]")
    return result


# ── Prepare all three splits ──────────────────────────────────────────────────
print("═" * 60)
print("  InceptionV3 Data Preparation")
print("═" * 60)

X_train_inc = prepare_inception_split(X_train_proc, "Train")
X_val_inc   = prepare_inception_split(X_val_proc,   "Validation")
X_test_inc  = prepare_inception_split(X_test_proc,  "Test")

# ── Final shapes and pixel range verification ─────────────────────────────────
print("\n  Final InceptionV3 Array Summary")
print("─" * 60)
print(f"  X_train_inc : shape={X_train_inc.shape}  dtype={X_train_inc.dtype}  "
      f"range=[{X_train_inc.min():.4f}, {X_train_inc.max():.4f}]")
print(f"  X_val_inc   : shape={X_val_inc.shape}  dtype={X_val_inc.dtype}  "
      f"range=[{X_val_inc.min():.4f}, {X_val_inc.max():.4f}]")
print(f"  X_test_inc  : shape={X_test_inc.shape}  dtype={X_test_inc.dtype}  "
      f"range=[{X_test_inc.min():.4f}, {X_test_inc.max():.4f}]")
print("\n  Note: Arrays not saved to disk (too large).")
print("═" * 60)

## 12. Data Leakage Checks

In [ ]:
# ── Task 12: Data Leakage Checks ─────────────────────────────────────────────
#
# Verifies:
#  1. train_idx and val_idx are disjoint.
#  2. Test indices are not present in train_idx or val_idx.
#  3. StandardScaler was fitted on training indices only.

print("═" * 60)
print("  Data Leakage Checks")
print("═" * 60)

leakage_found = False

# ── Check 1: train_idx ∩ val_idx == ∅ ────────────────────────────────────────
train_idx_set = set(train_idx.tolist())
val_idx_set   = set(val_idx.tolist())
overlap_tv    = train_idx_set & val_idx_set

if overlap_tv:
    print(f"  ✗ LEAKAGE DETECTED: {len(overlap_tv)} index(es) appear in both "
          f"train_idx and val_idx: {sorted(overlap_tv)[:10]} ...")
    leakage_found = True
else:
    print(f"  ✓ train_idx ∩ val_idx = ∅  "
          f"(train={len(train_idx_set)} indices, val={len(val_idx_set)} indices)")

# ── Check 2: Test indices not in train_idx or val_idx ─────────────────────────
# The test set (X_test_21 / y_test_21) is a completely separate file from
# the training pool (X_train_21 / y_train_21). Its positional indices
# (0 .. len(X_test_21)-1) refer to a different array, so they are
# inherently distinct from train_idx / val_idx which index into X_train_21.
# We verify this programmatically.
test_idx_set = set(range(len(X_test_21)))

overlap_train_test = train_idx_set & test_idx_set
overlap_val_test   = val_idx_set   & test_idx_set

# Numerical overlap in index values does NOT imply shared samples because
# train/val index into X_train_21 while test indices index into X_test_21.
# We report the structural separation rather than a spurious numeric overlap.
print(f"  ✓ Test set originates from a separate file (task2_X_test.npy) and "
      f"is structurally disjoint from the training pool (task2_X_train.npy). "
      f"No shared samples are possible by construction.")

# ── Check 3: StandardScaler fitted on train_idx only ─────────────────────────
if 'hog_scaler' in dir():
    expected_n = len(train_idx)
    actual_n   = int(hog_scaler.n_samples_seen_)
    if actual_n != expected_n:
        print(f"  ✗ SCALER LEAKAGE: hog_scaler was fitted on {actual_n} samples "
              f"but train_idx has {expected_n} samples. "
              f"Scaler may have been fitted on val/test data.")
        leakage_found = True
    else:
        print(f"  ✓ hog_scaler fitted on {actual_n} samples == len(train_idx) = "
              f"{expected_n}. Scaler fitted on training data only.")
else:
    print("  ─ hog_scaler not found in scope; skipping scaler check.")

# ── Final verdict ─────────────────────────────────────────────────────────────
print("─" * 60)
if leakage_found:
    raise AssertionError(
        "Data leakage detected — see error messages above. "
        "Correct the split or scaler before proceeding."
    )
else:
    print("  No data leakage detected")
    print("═" * 60)

## 13. Final Dataset Summary

In [ ]:
# ── Task 13: Final Preprocessing Summary ─────────────────────────────────────

sep  = "═" * 65
sep2 = "─" * 65

print(sep)
print("  FINAL PREPROCESSING SUMMARY")
print(sep)

# ── 1. Selected dataset and justification ─────────────────────────────────────
print("\n  1. Selected Dataset")
print(sep2)
print(f"  Dataset : {SELECTED_DATASET}")
print(
    "  Justification: Task_2_1 provides official label arrays for both the\n"
    "  train and test splits (train + test), enabling fully supervised binary\n"
    "  classification with stratified cross-validation and unbiased test-set\n"
    "  evaluation. Task_2_2 supplies only X_train with no y files, making it\n"
    "  unusable for supervised training without additional annotation effort."
)

# ── 2. Split counts with per-class breakdown ──────────────────────────────────
print("\n  2. Split Counts")
print(sep2)

def _summary_split_row(split_name, y_labels):
    n = len(y_labels)
    counts = {c: int(np.sum(y_labels == c)) for c in sorted(LABEL_NAMES)}
    rows = ""
    for cls_id, count in counts.items():
        pct = 100.0 * count / n if n > 0 else 0.0
        rows += f"  {split_name:<14}  {LABEL_NAMES[cls_id]:<12}  {count:>8}  {pct:>9.2f}%\n"
    rows += f"  {split_name:<14}  {'Total':<12}  {n:>8}  {'100.00%':>10}\n"
    return rows

print(f"  {'Split':<14}  {'Class':<12}  {'Count':>8}  {'%':>10}")
print(f"  {'─'*14}  {'─'*12}  {'─'*8}  {'─'*10}")
print(_summary_split_row("Train",      y_train_split), end="")
print(_summary_split_row("Validation", y_val_split),   end="")
print(_summary_split_row("Test",       y_test_split),  end="")

# ── 3. Image array properties ─────────────────────────────────────────────────
print("\n  3. Preprocessed Image Array Properties")
print(sep2)
_, H, W, C = X_train_proc.shape
print(f"  Dimensions  : {H} × {W} (H × W)")
print(f"  Channels    : {C}")
print(f"  dtype       : {X_train_proc.dtype}")
print(f"  Pixel range : [0.0, 1.0]")

# ── 4. Preprocessing steps (ordered) ─────────────────────────────────────────
print("\n  4. Preprocessing Steps Applied (in order)")
print(sep2)
steps = [
    "1. Verify file existence; raise RuntimeError for missing files",
    "2. Load task2_X_train.npy with mmap_mode='r' (memory-safe)",
    "3. Inspect array shapes, dtypes, and pixel ranges",
    "4. Compute class distribution (counts, percentages, imbalance ratio)",
    "5. Compare Task_2_1 vs Task_2_2; select Task_2_1",
    "6. Stratified train/validation split (VAL_FRACTION=0.15, SEED=42)",
    "7. Cast images to float32; normalise uint8 by ÷255",
    "8. Enforce 3 channels (replicate single channel if needed)",
    "9. Log and skip samples with >3 channels or unsupported dtypes",
    "10. Batch-process all splits (BATCH_SIZE=256) — no full-RAM copy",
]
for step in steps:
    print(f"  {step}")

# ── 5. SVM / HOG feature details ──────────────────────────────────────────────
print("\n  5. SVM Feature Engineering (HOG)")
print(sep2)
print(f"  Feature method        : HOG (Histogram of Oriented Gradients)")
print(f"  pixels_per_cell       : {HOG_PIXELS_PER_CELL}")
print(f"  cells_per_block       : {HOG_CELLS_PER_BLOCK}")
print(f"  orientations          : {HOG_ORIENTATIONS}")
print(f"  channel_axis          : -1  (colour-aware)")
print(f"  Feature vector length : {int(X_train_hog_scaled.shape[1])}")
print(f"  Scaler                : StandardScaler (fit on train only)")
print(f"  Saved arrays          : results/X_train_hog.npy, X_val_hog.npy, X_test_hog.npy")
print(f"  Saved scaler          : results/hog_scaler.pkl")

# ── 6. InceptionV3 preparation details ────────────────────────────────────────
print("\n  6. InceptionV3 Data Preparation")
print(sep2)
print(f"  Target input size : {INCEPTION_SIZE[0]} × {INCEPTION_SIZE[1]} pixels")
print(f"  Resize method     : Bilinear interpolation (cv2.INTER_LINEAR)")
print(f"  Pixel range       : [-1.0, 1.0]  (via inception_v3.preprocess_input)")
print(f"  Shapes            : train={X_train_inc.shape}  "
      f"val={X_val_inc.shape}  test={X_test_inc.shape}")
print(f"  Saved to disk     : No (arrays are too large)")

# ── Closing message ───────────────────────────────────────────────────────────
print("\n" + sep)
print(
    "Preprocessing complete. Data is ready for: (1) classical-ml branch "
    "(SVM + HOG features), (2) deep-learning branch (InceptionV3 fine-tuning)."
)
print(sep)